# Gemma 4 E2B FFT training

Burstchester dataset ID를 받아 Gemma 4 E2B full fine-tuning을 실행하고, 선택적으로 Hugging Face 업로드와 Burstchester 모델 등록까지 수행한다.

In [ ]:
# 1. Clone the CLI repository.
!git clone https://github.com/tomongoose/burstchester.git /content/burstchester
%cd /content/burstchester

In [ ]:
# 2. Load secrets from Colab secrets first, then fall back to hidden input.
import os
from getpass import getpass

try:
    from google.colab import userdata
except Exception:
    userdata = None

def secret(name, prompt=None, required=True):
    value = os.environ.get(name)
    if not value and userdata is not None:
        try:
            value = userdata.get(name)
        except Exception:
            value = None
    if not value and prompt:
        value = getpass(prompt)
    if required and not str(value or '').strip():
        raise ValueError(f'{name} is required.')
    if value:
        os.environ[name] = str(value).strip()
    return os.environ.get(name, '')

secret('BURSTCHESTER_ACCESS_TOKEN', 'Burstchester access token: ')
secret('HF_TOKEN', 'Hugging Face token: ')
os.environ['HUGGING_FACE_HUB_TOKEN'] = os.environ['HF_TOKEN']
print('Secrets configured in environment.')

In [ ]:
# 3. Configure Gemma 4 E2B FFT.
import os

os.environ['DATASET_IDS'] = 'dataset-id-1,dataset-id-2'  # Change this.
os.environ['TRAIN_COMMAND'] = 'train-gemma4-e2b-full'
os.environ['BASE_MODEL'] = 'google/gemma-4-E2B'
os.environ['WORKSPACE'] = '/content/burstchester-training/gemma4-e2b-fft'
os.environ['OUTPUT_MODEL_REPO'] = 'hf-user/gemma4-e2b-fft'  # Change this.
os.environ['TRAINING_METHOD'] = 'full'
os.environ['EPOCHS'] = '1'
os.environ['BATCH_SIZE'] = '1'
os.environ['MAX_SEQ_LENGTH'] = '128'
os.environ['MODEL_POINT_COST'] = '100'

# Full fine-tuning needs a large GPU. Set SKIP_REGISTER=1 to train without registering.
# os.environ['SKIP_REGISTER'] = '1'

In [ ]:
# 4. Run training, upload, and registration.
!bash cli/scripts/colab-train-and-register.sh